In [ ]:
import os
import torch
import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Seed set.")


PosixPath('/home/octoopt/workspace/projects/competition/ZaloAI25')

In [ ]:
from predict import Model

model = Model("./saved_models/yolo11n.pt")

In [ ]:
import cv2
import glob

DATA_DIR = "/data/samples/*/drone_video.mp4"
test_videos = sorted(glob.glob(DATA_DIR))
print("Total test videos:", len(test_videos))


In [ ]:
import time
import json
import pandas as pd

all_results = []
all_time = []

for vid in test_videos:
    video_id = vid.split("/")[-2]
    print("Predicting:", video_id)

    cap = cv2.VideoCapture(vid)
    frame_idx = 0

    t0 = time.time()

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        bbox = model.predict_streaming(frame_rgb, frame_idx)
        
        frame_idx += 1

    t1 = time.time()
    total_ms = int((t1 - t0) * 1000)

    all_time.append({"id": video_id, "time": total_ms})

cap.release()

# ---- Write outputs ----

# JSON
os.makedirs("/result", exist_ok=True)

json_path = "/result/jupyter_submission.json"
time_path = "/result/jupyter_time_submission.csv"

# results identical to model.write_submission() but with jupyter_ prefix
json_data = []
for fid, bbox in model.results.items():
    json_data.append({"id": fid, "bbox": bbox})

with open(json_path, "w") as f:
    json.dump(json_data, f, indent=2)

pd.DataFrame(all_time).to_csv(time_path, index=False)

print("[SAVE] jupyter_submission.json")
print("[SAVE] jupyter_time_submission.csv")